In [1]:
import numpy as np
import pandas as pd


In [2]:
def bayes_theorem(k_probs, b_given_k):
    """Calculates posterior probabilities using Bayes' theorem."""
    # Compute marginal probability of B
    p_b = sum(k_probs[i] * b_given_k[i] for i in range(len(k_probs)))

    # Compute P(K_i | B) for each event K_i using Bayes' theorem
    p_k_given_b = [(k_probs[i] * b_given_k[i]) / p_b for i in range(len(k_probs))]

    return p_b, p_k_given_b


In [4]:
# Load dataset
file_path = "WHR2024.csv"
df = pd.read_csv(file_path)


In [5]:
# Define happiness categories (High, Medium, Low) based on Ladder Score
bins = [df["Ladder score"].min(), df["Ladder score"].median() - 0.1, df["Ladder score"].median() + 0.1, df["Ladder score"].max()]
labels = ["Low", "Medium", "High"]
df["Happiness Category"] = pd.cut(df["Ladder score"], bins=bins, labels=labels, include_lowest=True)

# Define Event B: High Social Support (above median value)
median_social_support = df["Explained by: Social support"].median()
df["High Social Support"] = df["Explained by: Social support"] > median_social_support

# Compute P(K_i) for each happiness category
k_probs = df["Happiness Category"].value_counts(normalize=True).reindex(labels, fill_value=0).tolist()


In [6]:
# Compute P(B | K_i) for each happiness category
b_given_k = []
for category in labels:
    subset = df[df["Happiness Category"] == category]
    if len(subset) > 0:
        b_given_k.append(subset["High Social Support"].mean())
    else:
        b_given_k.append(0)

# Compute probabilities using Bayes' theorem
p_b, p_k_given_b = bayes_theorem(k_probs, b_given_k)

In [7]:
# Print results
print(f"Computed Marginal Probability of High Social Support (B): {p_b}")
print("Computed P(Happiness Category | B):")
for i, prob in enumerate(p_k_given_b):
    print(f"P({labels[i]} | High Social Support) = {prob}")


Computed Marginal Probability of High Social Support (B): 0.4895104895104895
Computed P(Happiness Category | B):
P(Low | High Social Support) = 0.08571428571428573
P(Medium | High Social Support) = 0.14285714285714288
P(High | High Social Support) = 0.7714285714285715
